# DS-3002 <br> Data Mining - Assignment 2

**Muhammad Moiz Khalid** <br>
**23i-2552**<br>
**BDS-6C**

## Preprocessing

In [ ]:
import re, json, random, time, os
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.makedirs("embeddings", exist_ok=True)
print("Imports OK | PyTorch:", torch.__version__)


## Data Loading & Tokenisation

In [ ]:
def load_articles(filepath):
    """Parse numbered [N] articles from cleaned.txt / raw.txt."""
    with open(filepath, encoding='utf-8') as f:
        content = f.read()
    parts = re.split(r'\[(\d+)\]', content)
    articles = {}
    for i in range(1, len(parts), 2):
        text = parts[i + 1].strip()
        if text:
            articles[int(parts[i])] = text
    return articles

def tokenize(text):
    """Keep only Urdu Unicode characters; split on whitespace."""
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    return [t for t in text.split() if t]

cleaned_articles = load_articles('cleaned.txt')
raw_articles = load_articles('raw.txt')
metadata = json.load(open('Metadata.json', encoding='utf-8'))

tokenized_cleaned = {aid: tokenize(t) for aid, t in cleaned_articles.items()}
tokenized_raw = {aid: tokenize(t) for aid, t in raw_articles.items()}

doc_ids = sorted(cleaned_articles.keys())

print(f"Cleaned articles : {len(cleaned_articles)}")
print(f"Raw articles : {len(raw_articles)}")
print(f"Metadata entries : {len(metadata)}")

### Topic Assignment (keyword-based, 5 categories)

In [ ]:
TOPIC_KEYWORDS = {
    'Politics':       ['الیکشن', 'حکومت', 'وزیر', 'پارلیمنٹ', 'سیاست',
                       'جماعت', 'ووٹ', 'وزیراعظم', 'صدر', 'اسمبلی'],
    'Sports':         ['کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈکپ',
                       'فٹبال', 'کھیل', 'ٹورنامنٹ', 'وکٹ', 'گول'],
    'Economy':        ['مہنگائی', 'تجارت', 'بینک', 'بجٹ', 'معیشت',
                       'روپیہ', 'قرض', 'سرمایہ', 'ٹیکس', 'ڈالر'],
    'International':  ['اقوام', 'معاہدہ', 'امریکہ', 'چین', 'ایران',
                       'یوکرین', 'اسرائیل', 'روس', 'غیرملکی', 'تنازع'],
    'Health_Society': ['ہسپتال', 'بیماری', 'ویکسین', 'سیلاب', 'تعلیم',
                       'صحت', 'وبا', 'ڈاکٹر', 'آفت', 'امداد'],
}

def assign_topic(article_id):
    combined = (metadata.get(str(article_id), {}).get('title', '') +
                cleaned_articles.get(article_id, ''))
    scores = {t: sum(combined.count(k) for k in kws)
              for t, kws in TOPIC_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'International'

article_topics = {aid: assign_topic(aid) for aid in cleaned_articles}

print("Topic distribution:")
for t, c in sorted(Counter(article_topics.values()).items()):
    print(f"  {t:<20}: {c}")


## 1.1 TF-IDF Weighting

In [ ]:
MAX_VOCAB = 10_000

all_tokens = [tok for tokens in tokenized_cleaned.values() for tok in tokens]
token_freq = Counter(all_tokens)

top_vocab = [w for w, _ in token_freq.most_common(MAX_VOCAB)]
vocab     = {w: i for i, w in enumerate(['<UNK>'] + top_vocab)}
inv_vocab = {i: w for w, i in vocab.items()}

V = len(vocab)          # vocabulary size (incl. <UNK>)
D = len(cleaned_articles)
print(f"Vocabulary size (incl. <UNK>): {V}")
print(f"Number of documents          : {D}")


In [ ]:
# ── Term-Document matrix (raw counts) shape: V × D ──
print("Building term-document matrix ...")
word_doc_counts = np.zeros((V, D), dtype=np.float32)
for d_idx, aid in enumerate(doc_ids):
    for tok in tokenized_cleaned[aid]:
        word_doc_counts[vocab.get(tok, 0), d_idx] += 1

# TF: normalised by document length
doc_lengths = word_doc_counts.sum(axis=0, keepdims=True)
doc_lengths[doc_lengths == 0] = 1
TF = word_doc_counts / doc_lengths          # V × D

# DF & IDF
DF  = (word_doc_counts > 0).sum(axis=1)    # V
IDF = np.log(D / (1 + DF))                 # V  (standard formula from assignment)

# TF-IDF
tfidf_matrix = TF * IDF[:, np.newaxis]     # V × D

np.save('embeddings/tfidf_matrix.npy', tfidf_matrix)
print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print("Saved → embeddings/tfidf_matrix.npy")


In [ ]:
print("Top-10 most discriminative words per topic (mean TF-IDF):\n")
for topic in TOPIC_KEYWORDS:
    topic_doc_indices = [i for i, aid in enumerate(doc_ids)
                         if article_topics[aid] == topic]
    if not topic_doc_indices:
        continue
    mean_tfidf = tfidf_matrix[:, topic_doc_indices].mean(axis=1)
    top10_ids  = np.argsort(mean_tfidf)[::-1][:15]
    top10_words = [inv_vocab[i] for i in top10_ids
                   if inv_vocab[i] != '<UNK>'][:10]
    print(f"  {topic}:")
    for rank, w in enumerate(top10_words, 1):
        print(f"    {rank:2d}. {w}")
    print()
